In [1]:
"""
EnterpriseRAG-Bench exploration for choosing a chunking strategy
Requires: pip install datasets pandas tiktoken matplotlib seaborn
"""

from __future__ import annotations

import random
from collections import Counter, defaultdict
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import tiktoken


# ================================================================
# 0. Setup
# ================================================================

random.seed(42)

DOCUMENTS_PATH = Path("data/documents/test.parquet") 
QUESTIONS_PATH = Path("data/questions/test.parquet")

OUTPUT_DIR = Path("artifacts/enterprise_rag")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Number of documents to retain per source for inspection/statistics.
MAX_PER_SOURCE = 800

# Number of rows processed from Parquet at a time.
BATCH_SIZE = 5_000

# Tokenizer
enc = tiktoken.get_encoding("cl100k_base")


def count_tokens(text: str) -> int:
    """Count tokens using the cl100k_base tokenizer."""
    if not text or not isinstance(text, str):
        return 0
    return len(enc.encode(text))

In [2]:
# ================================================================
# 1. Validate input files
# ================================================================

if not DOCUMENTS_PATH.exists():
    raise FileNotFoundError(f"Documents file not found:\n{DOCUMENTS_PATH}")

if not QUESTIONS_PATH.exists():
    raise FileNotFoundError(f"Questions file not found:\n{QUESTIONS_PATH}")

print("=" * 70)
print("EnterpriseRAG-Bench Exploration")
print("=" * 70)

print(f"\nDocuments: {DOCUMENTS_PATH}")
print(f"Questions: {QUESTIONS_PATH}")

# ================================================================
# 2. Load questions
# ================================================================
# Only ~500 questions exist, so loading them completely into memory
# is perfectly fine. We use pandas here because it makes filtering easy.

print("\nLoading questions...")

questions_df = pd.read_parquet(QUESTIONS_PATH)
print(f"Questions: {len(questions_df):,}")

# Convert to dictionaries for convenient access later.
questions = questions_df.to_dict("records")


# ================================================================
# 3. Open documents Parquet file
# ================================================================
# pq.read_table(...) would load the entire table into memory.
# Instead, use iter_batches(...) so only a small batch exists in memory.

print("\nOpening documents Parquet file...")

documents_file = pq.ParquetFile(DOCUMENTS_PATH)

print(f"Documents: {documents_file.metadata.num_rows:,}")
print(f"Row groups: {documents_file.num_row_groups:,}")

# Only read the columns we actually need.
DOCUMENT_COLUMNS = ["doc_id", "source_type", "title", "content"]

EnterpriseRAG-Bench Exploration

Documents: data/documents/test.parquet
Questions: data/questions/test.parquet

Loading questions...
Questions: 500

Opening documents Parquet file...
Documents: 511,962
Row groups: 1


In [5]:
from typing import List, Optional, Callable, Dict, Any
import re
import tiktoken
from helpers.config import get_settings

from langchain_text_splitters import (
    RecursiveCharacterTextSplitter,
    MarkdownHeaderTextSplitter,
)

ENCODING = tiktoken.get_encoding("cl100k_base")


def token_length(text: str) -> int:
    """Consistent token count (cl100k_base)."""
    if not text:
        return 0
    return len(ENCODING.encode(text, disallowed_special=()))


# --------------------------------------------------------------------
# Base + factory
# --------------------------------------------------------------------

class BaseChunker:
    strategy_name: str = "base"
    settings = get_settings()

    def chunk(self, text: str) -> List[str]:
        raise NotImplementedError


class RecursiveChunker(BaseChunker):
    strategy_name = "recursive"

    def __init__(
        self,
        chunk_size: Optional[int] = None,
        chunk_overlap: Optional[int] = None,
        length_function: Callable[[str], int] = token_length,
    ):
        self.chunk_size = chunk_size if chunk_size is not None else self.settings.RECURSIVE_CHUNK_SIZE
        self.chunk_overlap = chunk_overlap if chunk_overlap is not None else self.settings.RECURSIVE_CHUNK_OVERLAP
        if self.chunk_overlap >= self.chunk_size:
            raise ValueError("chunk_overlap must be smaller than chunk_size")

        self._splitter = RecursiveCharacterTextSplitter(
            chunk_size=self.chunk_size,
            chunk_overlap=self.chunk_overlap,
            length_function=length_function,
            separators=["\n\n", "\n", ". ", " ", ""],
        )

    def chunk(self, text: str) -> List[str]:
        if not text or not text.strip():
            return []
        return self._splitter.split_text(text)


# --------------------------------------------------------------------
# Source-aware chunkers
# --------------------------------------------------------------------

class FirefliesChunker(BaseChunker):
    strategy_name = "fireflies"

    def __init__(self, length_function: Callable[[str], int] = token_length):
        self.chunk_size = self.settings.FIREFLIES_CHUNK_SIZE
        self.chunk_overlap = self.settings.FIREFLIES_CHUNK_OVERLAP
        self.length_function = length_function
        self._fallback = RecursiveChunker(
            chunk_size=self.chunk_size,
            chunk_overlap=self.chunk_overlap,
            length_function=length_function,
        )

    def chunk(self, text: str) -> List[str]:
        if not text or not text.strip():
            return []

        chunks: List[str] = []
        remaining = text

        summary_match = re.search(
            r"(?is)^(summary\s*:?\s*)(.*?)(?=\n\s*(transcript|topics|action_items)\s*:|$)",
            text,
        )
        if summary_match:
            summary = summary_match.group(0).strip()
            if summary:
                chunks.append(summary)
            remaining = text[summary_match.end():].strip()

        parts = re.split(r"(?=\[\d{1,2}:\d{2}(?::\d{2})?\])", remaining)
        parts = [p.strip() for p in parts if p and p.strip()]

        if not parts:
            return chunks or self._fallback.chunk(text)

        current: List[str] = []
        current_len = 0
        for part in parts:
            part_len = self.length_function(part)
            if current and current_len + part_len > self.chunk_size:
                chunks.append("\n".join(current))
                if self.chunk_overlap > 0 and current:
                    current = [current[-1]]
                    current_len = self.length_function(current[0])
                else:
                    current = []
                    current_len = 0
            current.append(part)
            current_len += part_len

        if current:
            chunks.append("\n".join(current))

        final: List[str] = []
        for c in chunks:
            if self.length_function(c) > self.chunk_size * 1.5:
                final.extend(self._fallback.chunk(c))
            else:
                final.append(c)
        return final


class ConfluenceChunker(BaseChunker):
    strategy_name = "confluence"

    def __init__(self, length_function: Callable[[str], int] = token_length):
        self.chunk_size = self.settings.CONFLUENCE_CHUNK_SIZE
        self.chunk_overlap = self.settings.CONFLUENCE_CHUNK_OVERLAP
        self.length_function = length_function
        self._headers = [("#", "h1"), ("##", "h2"), ("###", "h3")]
        self._fallback = RecursiveChunker(
            chunk_size=self.chunk_size,
            chunk_overlap=self.chunk_overlap,
            length_function=self.length_function,
        )

    def chunk(self, text: str) -> List[str]:
        if not text or not text.strip():
            return []

        if not re.search(r"(?m)^#{1,3}\s+\S", text) and not re.search(
            r"(?m)^(Overview|Goal and scope|Purpose|Scope|Terminology|Background)\s*$",
            text,
        ):
            return self._fallback.chunk(text)

        try:
            from langchain_text_splitters import MarkdownHeaderTextSplitter
            md_splitter = MarkdownHeaderTextSplitter(headers_to_split_on=self._headers)
            docs = md_splitter.split_text(text)
            sections = []
            for d in docs:
                header_bits = [f"{k}: {v}" for k, v in (d.metadata or {}).items()]
                prefix = (" | ".join(header_bits) + "\n") if header_bits else ""
                sections.append(prefix + d.page_content)
        except Exception:
            return self._fallback.chunk(text)

        out: List[str] = []
        for sec in sections:
            if self.length_function(sec) <= self.chunk_size:
                out.append(sec)
            else:
                out.extend(self._fallback.chunk(sec))
        return out or self._fallback.chunk(text)

class SlackChunker(BaseChunker):
    strategy_name = "slack"

    MSG_RE = re.compile(
        r"(?m)^([A-Za-z0-9_./\-]+(?:\s+[A-Za-z0-9_./\-]+)*):\s*(.*?)(?=^\S[^:\n]*:|\Z)",
        re.DOTALL,
    )

    def __init__(self, length_function: Callable[[str], int] = token_length):
        self.chunk_size = self.settings.SLACK_CHUNK_SIZE
        self.chunk_overlap = self.settings.SLACK_CHUNK_OVERLAP
        self.length_function = length_function
        self._fallback = RecursiveChunker(
            chunk_size=self.chunk_size,
            chunk_overlap=self.chunk_overlap,
            length_function=self.length_function,
        )

    def chunk(self, text: str) -> List[str]:
        if not text or not text.strip():
            return []

        messages = [f"{m.group(1)}: {m.group(2).strip()}" for m in self.MSG_RE.finditer(text)]

        if len(messages) < 2:
            return self._fallback.chunk(text)

        chunks: List[str] = []
        current: List[str] = []
        current_len = 0
        for msg in messages:
            msg_len = self.length_function(msg)
            if current and current_len + msg_len > self.chunk_size:
                chunks.append("\n".join(current))
                if self.chunk_overlap > 0 and current:
                    current = [current[-1]]
                    current_len = self.length_function(current[0])
                else:
                    current = []
                    current_len = 0
            current.append(msg)
            current_len += msg_len + 1

        if current:
            chunks.append("\n".join(current))
        return chunks


class TicketChunker(BaseChunker):
    strategy_name = "ticket"

    def __init__(self, length_function: Callable[[str], int] = token_length):
        self.chunk_size = self.settings.TICKET_CHUNK_SIZE
        self.chunk_overlap = self.settings.TICKET_CHUNK_OVERLAP
        self.length_function = length_function
        self._fallback = RecursiveChunker(
            chunk_size=self.chunk_size,
            chunk_overlap=self.chunk_overlap,
            length_function=self.length_function,
        )

    def chunk(self, text: str) -> List[str]:
        if not text or not text.strip():
            return []

        blocks = re.split(
            r"(?i)(?=\n(?:description|comments|acceptance_criteria|body)\s*:)",
            text,
        )
        blocks = [b.strip() for b in blocks if b and b.strip()]

        if len(blocks) <= 1:
            return self._fallback.chunk(text)

        out: List[str] = []
        for block in blocks:
            if self.length_function(block) <= self.chunk_size:
                out.append(block)
            else:
                out.extend(self._fallback.chunk(block))
        return out


class GmailChunker(BaseChunker):
    strategy_name = "gmail"

    REPLY_BOUNDARY = re.compile(
        r"(?m)^(?:"
        r"On .+ wrote:|"
        r"From:\s*.+$|"
        r"-{2,}\s*Original Message\s*-{2,}"
        r")",
    )

    def __init__(self, length_function: Callable[[str], int] = token_length):
        self.chunk_size = self.settings.GMAIL_CHUNK_SIZE
        self.chunk_overlap = self.settings.GMAIL_CHUNK_OVERLAP
        self.length_function = length_function
        self._fallback = RecursiveChunker(
            chunk_size=self.chunk_size,
            chunk_overlap=self.chunk_overlap,
            length_function=self.length_function,
        )

    def chunk(self, text: str) -> List[str]:
        if not text or not text.strip():
            return []

        if self.length_function(text) <= self.chunk_size:
            return [text.strip()]

        parts = self.REPLY_BOUNDARY.split(text)
        parts = [p.strip() for p in parts if p and p.strip()]
        if len(parts) <= 1:
            return self._fallback.chunk(text)

        out: List[str] = []
        current = ""
        for part in parts:
            candidate = (current + "\n\n" + part).strip() if current else part
            if current and self.length_function(candidate) > self.chunk_size:
                out.append(current)
                current = part
            else:
                current = candidate
        if current:
            out.append(current)

        final: List[str] = []
        for c in out:
            if self.length_function(c) > self.chunk_size * 1.4:
                final.extend(self._fallback.chunk(c))
            else:
                final.append(c)
        return final


# --------------------------------------------------------------------
# Router & Factory
# --------------------------------------------------------------------

class ChunkerFactory:
    _strategies: Dict[str, Any] = {
        "recursive": RecursiveChunker,
        "fireflies": FirefliesChunker,
        "confluence": ConfluenceChunker,
        "slack": SlackChunker,
        "ticket": TicketChunker,
        "gmail": GmailChunker,
    }

    # Map external source names/aliases to target strategy names
    SOURCE_MAP: Dict[str, str] = {
        "fireflies": "fireflies",
        "confluence": "confluence",
        "slack": "slack",
        "jira": "ticket",
        "linear": "ticket",
        "github": "ticket",
        "gmail": "gmail",
        "google_drive": "confluence",
        "gdrive": "confluence",
        "hubspot": "recursive",
    }

    @classmethod
    def _normalize_source(cls, source: Optional[str]) -> str:
        """Normalizes inputs like 'Jira-Ticket' or 'Google Drive' to mapped strategies."""
        if not source:
            return "recursive"
        
        # Clean spacing, dashes, and lowercase
        clean_key = re.sub(r"[\s\-]+", "_", source.strip().lower())
        return cls.SOURCE_MAP.get(clean_key, "recursive")

    @classmethod
    def get_chunker(
        cls, 
        source: Optional[str] = None, 
        strategy: Optional[str] = None,
        length_function: Callable[[str], int] = token_length,
        **kwargs
    ) -> BaseChunker:
        """
        Returns a chunker instance by strategy name or source type.
        """
        target_strategy = strategy or cls._normalize_source(source)
        chunker_cls = cls._strategies.get(target_strategy, RecursiveChunker)
        return chunker_cls(length_function=length_function, **kwargs)

    @classmethod
    def chunk(
        cls, 
        text: str, 
        source: Optional[str] = None, 
        strategy: Optional[str] = None, 
        **kwargs
    ) -> List[str]:
        """
        Direct one-shot helper to select chunker and split text in one line.
        """
        chunker = cls.get_chunker(source=source, strategy=strategy, **kwargs)
        return chunker.chunk(text)

In [7]:
chunker = ChunkerFactory.get_chunker(source="fireflies")
chunker.__class__

__main__.FirefliesChunker

In [4]:
# ============================================================
# Test source-aware chunkers on random EnterpriseRAG-Bench docs
# ============================================================
# pip install datasets tiktoken langchain-text-splitters pandas

from datasets import load_dataset
import random
import pandas as pd
from collections import defaultdict

# ---- import your chunkers (adjust path if needed) ----
# from your_module import (
#     ChunkerFactory, token_length, FirefliesChunker, ConfluenceChunker,
#     SlackChunker, TicketChunker, GmailChunker, RecursiveChunker
# )

# If the classes live in the same notebook, just ensure they are defined above.
# Otherwise uncomment the import.

random.seed(42)


# ----------------------------------------------------------
# 1. Load a manageable sample of documents
# ----------------------------------------------------------
# print("Loading documents (streaming / cached)...")
# docs = load_dataset("onyx-dot-app/EnterpriseRAG-Bench", "documents", split="test")

# Collect up to N docs per source
N_PER_SOURCE = 5
SAMPLES_PER_TEST = 2          # how many docs to actually print per source
MAX_PREVIEW_CHARS = 900       # truncate long chunk previews

by_source = defaultdict(list)
for batch_idx, batch in enumerate(
    documents_file.iter_batches(batch_size=BATCH_SIZE, columns=DOCUMENT_COLUMNS),
    start=1,
):
    rows = batch.to_pylist()

    for doc in rows:
        
        src = (doc.get("source_type") or "").lower()
        if len(by_source[src]) < N_PER_SOURCE:
            by_source[src].append(doc)
        if all(len(v) >= N_PER_SOURCE for v in by_source.values()) and len(by_source) >= 9:
            break

print("Samples available:")
for src, items in sorted(by_source.items(), key=lambda x: -len(x[1])):
    print(f"  {src:15s}: {len(items)}")



/home/youssef/miniconda3/envs/mini-rag-app/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Samples available:
  confluence     : 5
  fireflies      : 5
  github         : 5
  gmail          : 5
  google_drive   : 5
  hubspot        : 5
  jira           : 5
  linear         : 5
  slack          : 5


In [3]:
chunker = ChunkerFactory.get_chunker("auto", source_type="fireflies")
chunker.__class__

__main__.SourceAwareChunker

In [ ]:
# ----------------------------------------------------------
# 2. Strategy → default chunker kwargs (aligned with earlier advice)
# ----------------------------------------------------------
STRATEGY_CONFIG = {
    "fireflies":  dict(strategy="fireflies",  chunk_size=550, chunk_overlap=80),
    "confluence": dict(strategy="confluence", chunk_size=650, chunk_overlap=80),
    "slack":      dict(strategy="slack",      chunk_size=500, chunk_overlap=50),
    "jira":       dict(strategy="ticket",     chunk_size=550, chunk_overlap=60),
    "linear":     dict(strategy="ticket",     chunk_size=550, chunk_overlap=60),
    "github":     dict(strategy="ticket",     chunk_size=550, chunk_overlap=60),
    "gmail":      dict(strategy="gmail",      chunk_size=800, chunk_overlap=80),
    "google_drive": dict(strategy="confluence", chunk_size=600, chunk_overlap=80),
    "hubspot":    dict(strategy="recursive",  chunk_size=500, chunk_overlap=50),
}

# Also always compare against plain recursive
RECURSIVE_CFG = dict(strategy="recursive", chunk_size=500, chunk_overlap=50)

def get_chunker(cfg):
    return ChunkerFactory.get_chunker(**cfg)

def preview(text, max_chars=MAX_PREVIEW_CHARS):
    text = text or ""
    if len(text) <= max_chars:
        return text
    return text[:max_chars] + "\n... [truncated] ..."

# ----------------------------------------------------------
# 3. Run tests
# ----------------------------------------------------------
results_summary = []

for source, docs_list in sorted(by_source.items()):
    if source not in STRATEGY_CONFIG:
        print(f"\n[skip] no dedicated config for source={source}")
        continue

    cfg = STRATEGY_CONFIG[source]
    specialist = get_chunker(cfg)
    recursive = get_chunker(RECURSIVE_CFG)

    test_docs = random.sample(docs_list, min(SAMPLES_PER_TEST, len(docs_list)))

    print("\n" + "=" * 80)
    print(f"SOURCE: {source.upper()}  |  specialist={cfg['strategy']}  "
          f"(size={cfg['chunk_size']}, overlap={cfg['chunk_overlap']})")
    print("=" * 80)

    for i, doc in enumerate(test_docs, 1):
        content = doc.get("content") or ""
        title = (doc.get("title") or "")[:100]
        doc_id = doc.get("doc_id")
        n_tok = token_length(content)

        print(f"\n--- doc {i}/{len(test_docs)} | {doc_id}")
        print(f"Title : {title}")
        print(f"Length: {len(content):,} chars  ≈  {n_tok:,} tokens")

        # Specialist
        try:
            spec_chunks = specialist.chunk(content)
        except Exception as e:
            spec_chunks = []
            print(f"  SPECIALIST ERROR: {e}")

        # Recursive baseline
        try:
            rec_chunks = recursive.chunk(content)
        except Exception as e:
            rec_chunks = []
            print(f"  RECURSIVE ERROR: {e}")

        print(f"  Specialist → {len(spec_chunks)} chunks")
        print(f"  Recursive  → {len(rec_chunks)} chunks")

        # Token stats per strategy
        if spec_chunks:
            spec_toks = [token_length(c) for c in spec_chunks]
            print(f"  Specialist token sizes: min={min(spec_toks)}, "
                  f"median={sorted(spec_toks)[len(spec_toks)//2]}, max={max(spec_toks)}")
        if rec_chunks:
            rec_toks = [token_length(c) for c in rec_chunks]
            print(f"  Recursive  token sizes: min={min(rec_toks)}, "
                  f"median={sorted(rec_toks)[len(rec_toks)//2]}, max={max(rec_toks)}")

        # Show first 2 specialist chunks
        for j, ch in enumerate(spec_chunks[:2], 1):
            print(f"\n  [specialist chunk {j}/{len(spec_chunks)} | {token_length(ch)} tokens]")
            print("  " + preview(ch).replace("\n", "\n  "))

        if len(spec_chunks) > 2:
            print(f"\n  ... ({len(spec_chunks) - 2} more specialist chunks omitted)")

        results_summary.append({
            "source": source,
            "doc_id": doc_id,
            "tokens": n_tok,
            "specialist_strategy": cfg["strategy"],
            "n_specialist": len(spec_chunks),
            "n_recursive": len(rec_chunks),
            "spec_max_tok": max(spec_toks) if spec_chunks else None,
            "rec_max_tok": max(rec_toks) if rec_chunks else None,
        })

# ----------------------------------------------------------
# 4. Summary table
# ----------------------------------------------------------
df = pd.DataFrame(results_summary)
print("\n\n" + "=" * 80)
print("SUMMARY")
print("=" * 80)
display(df)  # in Jupyter; use print(df) in plain Python

# Optional: aggregate by source
agg = (
    df.groupby("source")
      .agg(
          n_docs=("doc_id", "count"),
          avg_doc_tokens=("tokens", "mean"),
          avg_specialist_chunks=("n_specialist", "mean"),
          avg_recursive_chunks=("n_recursive", "mean"),
      )
      .round(1)
)
print("\nBy source:")
display(agg)


SOURCE: CONFLUENCE  |  specialist=confluence  (size=650, overlap=80)

--- doc 1/2 | dsid_e54ef48bae78474684a957cf613d47d5
Title : Runbook: Deploy / Upgrade / Roll Back perf-canary (Prod)
Length: 12,894 chars  ≈  2,835 tokens
  Specialist → 35 chunks
  Recursive  → 7 chunks
  Specialist token sizes: min=25, median=78, max=269
  Recursive  token sizes: min=249, median=446, max=460

  [specialist chunk 1/35 | 120 tokens]
  h2: Purpose
  This runbook describes the operational procedures to deploy, upgrade, roll back, and safely disable the **perf-canary** service across regions.  
  perf-canary is a lightweight, always-on synthetic workload that calls internal inference endpoints for a curated model set and emits performance metrics (p50/p95/p99, tokens/sec, prefill/decode, queueing time, batch stats). The service must stay under the defined overhead budget and must not capture or emit customer data.  
  **Primary users:** Eng-Infra on-call and Release Engineering during rollouts.  
  ---

,source,doc_id,tokens,specialist_strategy,n_specialist,n_recursive,spec_max_tok,rec_max_tok
0,confluence,dsid_e54ef48bae78474684a957cf613d47d5,2835,confluence,35,7,269,460
1,confluence,dsid_d511c6d8daf94f998ce6a3d97462af2d,2325,confluence,5,6,613,491
2,fireflies,dsid_a94ba3c00ddc4c07bf685acd129b10ba,4851,fireflies,11,12,550,484
3,fireflies,dsid_9ed01291d81947f49500df5fb28ab724,8231,fireflies,17,19,549,480
4,github,dsid_72240c1112fd4f36aa172a84b9de1637,1254,ticket,4,3,515,440
5,github,dsid_61a7f1a19ac444ed8ed02f4b11e8e3de,1097,ticket,3,3,466,466
6,gmail,dsid_c6a6396159964deeb021d646eaaf2b0c,2730,gmail,4,7,769,489
7,gmail,dsid_fa0033ba74a14677849f30f943277542,1294,gmail,2,4,656,374
8,google_drive,dsid_291a3ea3d9da4d1eac54675268c41f39,1259,confluence,3,5,596,499
9,google_drive,dsid_5d6b7de2a9b7456e951649a82c6d861f,1549,confluence,4,4,567,485



By source:


,n_docs,avg_doc_tokens,avg_specialist_chunks,avg_recursive_chunks
source,,,,
confluence,2,2580.0,20.0,6.5
fireflies,2,6541.0,14.0,15.5
github,2,1175.5,3.5,3.0
gmail,2,2012.0,3.0,5.5
google_drive,2,1404.0,3.5,4.5
hubspot,2,538.5,1.5,1.5
jira,2,1594.5,4.5,4.0
linear,2,774.0,2.0,2.0
slack,2,762.0,2.0,2.0


In [24]:
def test_one(doc_id: str = None, source: str = "slack"):
    """Grab one random doc of a source (or by id) and print both strategies."""
    pool = by_source.get(source, [])
    # if doc_id:
    #     doc = next((d for d in docs if d["doc_id"] == doc_id), None)
    # else:
    doc = random.choice(pool) if pool else None
    if not doc:
        print("Not found")
        return

    content = doc["content"] or ""
    cfg = STRATEGY_CONFIG.get(source, RECURSIVE_CFG)
    specialist = get_chunker(cfg)
    chunks = specialist.chunk(content)

    print(f"{doc['doc_id']} | {source} | {doc['title']}| {token_length(content)} tokens → {len(chunks)} chunks\n")
    print(content)
    print("-----------------------------------------------------------\n")
    for i, c in enumerate(chunks, 1):
        print(f"----- chunk {i} ({token_length(c)} tok) -----")
        print(c)
        print()

# Examples:
# test_one(source="fireflies")
# test_one(source="confluence")
# test_one(doc_id="dsid_....")

In [25]:
test_one(source="confluence")

dsid_229dd48e9b1d466a81ebaffe3ec84469 | confluence | Cross-account GPU burst SLO contract and CI lifecycle| 2154 tokens → 4 chunks

Summary

Overview:
This document defines the operational contract (SLOs, quotas, billing guardrails) and the CI/CD lifecycle for the cross-account GPU burst fabric. It is intended for platform engineers, tenant owners, and on-call responders who coordinate cross-account bursting between Redwood-managed GPU pools and partner or tenant cloud accounts.

Goals and scope:
- Define a minimal, enforceable SLO contract that tenant teams and platform teams can rely on during bursts.
- Specify Terraform/IaC module shape and CI/CD checks required for enrolling an account in cross-account bursting.
- Provide an operator runbook for common failure modes (burst denied, egress exhaustion, billing disputes).
- Describe test, canary, and rollback requirements for rollouts that change allocation or billing behavior.

Key definitions:
- Burst: short-lived use of extra GPU ca

In [6]:
pool = by_source.get(source, [])
random.choice(pool)

{'doc_id': 'dsid_b9bc45edda744c5882658d381c41ae3d',
 'source_type': 'slack',
 'title': 'eng-platform',
 'content': 'maya: Quick async standup for API gateway + routing policies + auth + rate limits. Drop 1) what you did yesterday 2) blockers 3) plan for today. I’ll start.\n\nliam: 1) landed the new route priority selector for canary traffic, pushed to staging. 2) blocker: noticed increased 429s for a specific client when bursty traffic hits the same shard. 3) today: investigate burst handling (token bucket params) and tune shard routing.\n\nsasha: 1) finished the auth middleware change to surface client_id in headers for downstream services. Deployed to staging. 2) blocker: intermittent refresh_token failures on /v1/stream routes — reproduces when sequence > 4096 tokens. Logs show 401 from auth service. 3) today: run deeper trace with xsrf headers and add extra logging around token refresh path.\n\njordan: 1) reviewed liam’s routing unit tests and left comments. 2) blocker: holding on 

In [2]:
from openai import OpenAI

client = OpenAI(
    api_key="ollama",
    base_url="http://172.31.48.1:11434/v1"
)

response = client.chat.completions.create(
    model="qwen3.5:4b",
    messages=[
        {"role": "user", "content": "Hello"}
    ]
)

print(response.choices[0].message.content)

Hello! 👋 How can I help you today? Feel free to ask me anything or start a conversation!
